# BETE-NET 2-Structure Test
This notebook installs the exact PyTorch 1.10 environment required by BETE-NET via condacolab, clones the repository, formats the 2 candidates as CIFs, and runs the CSO ensemble.

In [ ]:
# Install condacolab to get Python 3.9 + conda on Google Colab
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# Install the exact pinned dependencies
import condacolab
condacolab.check()
!conda install -y python=3.9 pytorch==1.10.0 torchvision==0.11.0 torchaudio==0.10.0 cudatoolkit=11.3 -c pytorch -c conda-forge
!pip install ase==3.22.0 pymatgen==2024.8.9 scikit-learn==1.6.1
!pip install e3nn==0.4.2 torch-cluster==1.5.9 torch-scatter==2.0.9 torch-sparse==0.6.12 torch-spline-conv==1.2.1 torch-geometric==2.0.2 -f https://pytorch-geometric.com/whl/torch-1.10.0+cu113.html

In [ ]:
!git clone https://github.com/henniggroup/BETE-NET.git
%cd BETE-NET

In [ ]:
import os
import json
import time
import torch
import numpy as np
import pandas as pd
import ase.io
import torch_geometric as tg
from notebooks.utils.data import build_data, get_target, get_neighbors
from notebooks.utils.training import get_model

# Mock the database.json to just our two candidates
# Note: The CIFs must be manually uploaded to the 'structures/' directory.
test_candidates = ["mp-aaabfqpo", "mp-aaacpwsj"]

data_dicts = []
for idx in test_candidates:
    data_dicts.append({"index": idx, "target": np.zeros(51).tolist()})
df = pd.DataFrame(data_dicts)
df.to_json('database.json')
df.set_index('index', inplace=True)

In [ ]:
structures = []
for index, row in df.iterrows():
    structures.append(ase.io.read(f'structures/{index}.cif'))
df['structure'] = structures

df['target'] = df.apply(get_target, axis=1)
df['data'] = df.apply(build_data, embed_ph_dos=False, embed_e_dos=False, fine=False, r_max=4, axis=1)

device = "cuda:0" if torch.cuda.is_available() else "cpu"
out_dim = 51
in_dim = len(df.iloc[0].data.x[0])
em_dim = 64

init_dict_base = dict(in_dim=118, em_dim=em_dim, irreps_in=str(em_dim)+"x0e", 
    irreps_out=str(out_dim)+"x0e", irreps_node_attr=str(em_dim)+"x0e", 
    layers=2, mul=32, lmax=1, max_radius=4, 
    num_neighbors=get_neighbors(df, df.index).mean(), reduce_output=True, p=0.0)

dataloader = tg.loader.DataLoader(df['data'].values, batch_size=900)

In [ ]:
start_time = time.time()
for k in range(100):
    name = f"model_cso_{k}.pt"
    run_name = f'CSO/{name}'
    model, opt, scheduler = get_model(init_dict_base, device=device)
    model.load_state_dict(torch.load(run_name, map_location=device))
    model.pool = True
    model.to(device)
    model.eval()
    df[f'pred_{k}'] = np.empty((len(df), 1)).tolist()
    with torch.no_grad():
        for i, d in enumerate(dataloader):
            d.to(device)
            output = model(d)
            df[f'pred_{k}'] = [val for val in output.cpu().numpy()]
print(f"Ensemble inference took {time.time() - start_time:.2f} seconds")
print("GPU Memory Allocated:", torch.cuda.memory_allocated() / 1e9, "GB")